# Page classifier — BERT training on Colab

Runtime → Change runtime type → **GPU**. Edit `BASE_PATH` only.

The folder on Drive holds `bert-training/` (this folder, copied from the repo) and `training_data.jsonl`. The model and the resume state are written to Drive after every epoch, so a dropped session loses at most the epoch in progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = '/content/drive/MyDrive/page-classifier'  # <- the only thing to edit

DATA = f'{BASE_PATH}/training_data.jsonl'
OUTPUT = f'{BASE_PATH}/page-family'
RESUME = f'{BASE_PATH}/page-family-resume'
SCRIPT = f'{BASE_PATH}/bert-training/train.py'

In [ ]:
%pip install -q -r {BASE_PATH}/bert-training/requirements.txt

Compatibility check: two training steps and a time estimate. Saves nothing.

In [ ]:
!python "{SCRIPT}" --data "{DATA}" --output "{OUTPUT}" --resume-dir "{RESUME}" --check

A new run: 4 epochs (3–5 is a good range; the model folder keeps the best epoch, so more never hurts). Use `--restart` instead of nothing if a run already exists and you want to start over.

In [ ]:
!python "{SCRIPT}" --data "{DATA}" --output "{OUTPUT}" --resume-dir "{RESUME}" --epochs 4

One more epoch of the same run (same classes, same validation pages). Run again as often as the numbers keep improving.

In [ ]:
!python "{SCRIPT}" --data "{DATA}" --output "{OUTPUT}" --resume-dir "{RESUME}" --resume --epochs 1

In [ ]:
import pandas as pd
pd.read_csv(f'{OUTPUT}/training_log.csv')

In [ ]:
pd.read_csv(f'{OUTPUT}/category_wise_metrics.csv').sort_values('f1')

## Inference check

Try the saved model (the best epoch) on any text: the top 5 model types with probabilities, and the page type and codability each one means. This is BERT alone — the pipeline also runs the keyword model and the decision ladder, so its final answer can differ, especially for the model types BERT skipped.

In [ ]:
import sys
from pathlib import Path

CODE = Path(SCRIPT).parent  # the folder holding train.py, as the training cells use it
if not (CODE / 'predict.py').is_file():
    raise FileNotFoundError(f'predict.py is not in {CODE} - copy it there from training/bert-training, then rerun')
sys.path.insert(0, str(CODE))
from predict import Classifier

clf = Classifier(OUTPUT)
print(len(clf.labels), 'model types trained;', len(clf.untrained), 'left to the keyword model')

In [ ]:
# Paste a page's text and press Classify.
import ipywidgets as widgets
from IPython.display import display

box = widgets.Textarea(placeholder='Paste page text here', layout=widgets.Layout(width='100%', height='220px'))
button = widgets.Button(description='Classify', button_style='primary')
result = widgets.Output()

def on_click(_):
    with result:
        result.clear_output()
        clf.show(box.value)

button.on_click(on_click)
display(box, button, result)

In [ ]:
# No widgets? Paste the text between the triple quotes and run this cell.
TEXT = """
paste page text here
"""
clf.show(TEXT)